<h1>One-Hot-Encoding</h1>

<p>Dieses Skript tut alle Spalten, die weniger als 200 verschiedene Werte haben, One-Hot-Encoden. Spalten mit mehr verschiedenen Werten sind Freitextfelder und werden anders behandelt.</p>

In [1]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np

In [2]:
df = pd.read_csv("survey_results_cleaned_final.csv")
max_unique = 200
na_as_category = True
TOP_N = 10


In [3]:
df.LanguageHaveWorkedWith

0                         bash/shell (all shells);dart;sql
1                                                     java
2                      dart;html/css;javascript;typescript
3                                          java;kotlin;sql
4        c;c#;c++;delphi;html/css;java;javascript;lua;p...
                               ...                        
18598                javascript;python;ruby;sql;typescript
18599                       html/css;javascript;typescript
18600    bash/shell (all shells);html/css;java;powershe...
18601    bash/shell (all shells);delphi;powershell;pyth...
18602                                               c#;sql
Name: LanguageHaveWorkedWith, Length: 18603, dtype: object

In [4]:
multiselect_cols = [
    "LanguageHaveWorkedWith",
    "DatabaseHaveWorkedWith",
    "PlatformHaveWorkedWith",
    "WebframeHaveWorkedWith",
    "DevEnvsHaveWorkedWith",
    "OfficeStackAsyncHaveWorkedWith",
    "AIModelsHaveWorkedWith",
    "CommPlatformHaveWorkedWith",
    "AIAgent_Uses",
]
multiselect_cols = [c for c in multiselect_cols if c in df.columns]

SEP = ";"

def split_cell(x):
    # Missing/leer -> entweder [] oder ["none"]
    if pd.isna(x) or str(x).strip() == "":
        return ["none"] if na_as_category else []
    return [p.strip() for p in str(x).split(SEP) if p.strip()]

def multiselect_topN_with_other(df: pd.DataFrame, col: str, top_n: int = 10, na_as_category: bool = True):
    """
    Multi-Select (z.B. 'java;python;sql') -> Multi-Hot Encoding
    Behalte global Top-N Kategorien, bündle Rest in 'other'.
    Missing/leer -> 'none' (wenn na_as_category=True)
    """
    s = df[col].apply(split_cell)

    # Top-N global bestimmen (ohne 'none')
    exploded = s.explode()
    counts = exploded[exploded != "none"].value_counts()
    top = set(counts.head(top_n).index)

    def keep_top_and_other(items):
        if na_as_category and items == ["none"]:
            return ["none"]

        kept = [x for x in items if x in top]
        has_other = any((x not in top) and (x != "none") for x in items)

        out = set(kept)
        if has_other:
            out.add("other")
        if na_as_category and ("none" in items):
            out.add("none")
        return sorted(out)

    s2 = s.apply(keep_top_and_other)

    classes = sorted(top) + ["other"]
    if na_as_category:
        classes = ["none"] + classes

    mlb = MultiLabelBinarizer(classes=classes)
    dummies = pd.DataFrame(mlb.fit_transform(s2), columns=mlb.classes_, index=df.index)

    # Prefix wie vorher mit Spaltennamen
    dummies = dummies.add_prefix(f"{col}__")

    return dummies, sorted(top)

# --- Multi-Select: Top10 + other ---
for col in multiselect_cols:
    dummies, top10 = multiselect_topN_with_other(df, col, top_n=TOP_N, na_as_category=na_as_category)
    df = df.drop(columns=[col]).join(dummies)

    print(f"Top{TOP_N} für {col}: {top10}")

print("Form nach Multi-Select-OHE (Top10+other):", df.shape)

Top10 für LanguageHaveWorkedWith: ['bash/shell (all shells)', 'c#', 'c++', 'html/css', 'java', 'javascript', 'powershell', 'python', 'sql', 'typescript']
Top10 für DatabaseHaveWorkedWith: ['dynamodb', 'elasticsearch', 'mariadb', 'microsoft sql server', 'mongodb', 'mysql', 'oracle', 'postgresql', 'redis', 'sqlite']
Top10 für PlatformHaveWorkedWith: ['amazon web services (aws)', 'docker', 'google cloud', 'homebrew', 'kubernetes', 'microsoft azure', 'npm', 'pip', 'vite', 'yarn']
Top10 für WebframeHaveWorkedWith: ['angular', 'asp.net core', 'express', 'fastapi', 'jquery', 'next.js', 'node.js', 'react', 'spring boot', 'vue.js']
Top10 für DevEnvsHaveWorkedWith: ['android studio', 'cursor', 'intellij idea', 'jupyter notebook/jupyterlab', 'neovim', 'notepad++', 'pycharm', 'vim', 'visual studio', 'visual studio code']
Top10 für OfficeStackAsyncHaveWorkedWith: ['azure devops', 'confluence', 'github', 'gitlab', 'google workspace', 'jira', 'markdown file', 'miro', 'notion', 'obsidian']
Top10 für A

In [5]:
df

,Unnamed: 0,ResponseId,MainBranch,Age,EdLevel,Employment,Country,WorkExp,LearnCodeAI,YearsCode,...,AIAgent_Uses__customer service support,AIAgent_Uses__cybersecurity,AIAgent_Uses__data and analytics,AIAgent_Uses__decision intelligence,AIAgent_Uses__it operations,AIAgent_Uses__marketing,AIAgent_Uses__other industry purpose (write in):,AIAgent_Uses__robotics,AIAgent_Uses__software engineering,AIAgent_Uses__other
0,0,1,i am a developer by profession,25-34 years old,master’s degree,employed,Ukraine,8.0,"yes, i learned how to use ai-enabled tools for...",14.0,...,0,0,0,0,0,0,0,0,1,0
1,1,2,i am a developer by profession,25-34 years old,associate degree,employed,Netherlands,2.0,"yes, i learned how to use ai-enabled tools for...",10.0,...,0,0,0,0,0,0,0,0,0,0
2,2,3,i am a developer by profession,35-44 years old,bachelor’s degree,"independent contractor, freelancer, or self-em...",Ukraine,10.0,"yes, i learned how to use ai-enabled tools for...",12.0,...,0,0,0,0,0,0,0,0,1,0
3,3,4,i am a developer by profession,35-44 years old,bachelor’s degree,employed,Ukraine,4.0,"yes, i learned how to use ai-enabled tools for...",5.0,...,0,0,0,0,0,0,0,0,1,0
4,4,5,i am a developer by profession,35-44 years old,master’s degree,"independent contractor, freelancer, or self-em...",Ukraine,21.0,"yes, i learned how to use ai-enabled tools for...",22.0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18598,49017,49018,i am a developer by profession,18-24 years old,master’s degree,"independent contractor, freelancer, or self-em...",Ukraine,5.0,"yes, i learned how to use ai-enabled tools for...",6.0,...,0,0,0,0,0,0,0,0,0,0
18599,49066,49067,i am a developer by profession,35-44 years old,bachelor’s degree,employed,Australia,19.0,"yes, i learned how to use ai-enabled tools req...",22.0,...,0,0,0,0,0,0,0,0,0,0
18600,49074,49075,i am a developer by profession,25-34 years old,master’s degree,employed,India,5.0,"yes, i learned how to use ai-enabled tools for...",6.0,...,0,0,0,0,0,0,0,0,0,0
18601,49106,49107,i am a developer by profession,45-54 years old,bachelor’s degree,employed,South Africa,29.0,"yes, i learned how to use ai-enabled tools req...",29.0,...,0,0,0,0,0,0,0,0,0,0


In [6]:


print("Ursprüngliche DataFrame-Form:", df.shape)


Ursprüngliche DataFrame-Form: (18603, 137)


In [7]:

orgsize_order = [
    "just me - i am a freelancer, sole proprietor, etc.",
    "less than 20 employees",
    "20 to 99 employees",
    "100 to 499 employees",
    "500 to 999 employees",
    "1,000 to 4,999 employees",
    "5,000 to 9,999 employees",
    "10,000 or more employees",
]
orgsize_unknown = {"i don’t know"}

if "OrgSize" in df.columns:
    orgsize_map = {cat: i for i, cat in enumerate(orgsize_order)}
    max_val = len(orgsize_order) - 1

    df["OrgSize"] = (
        df["OrgSize"]
        .replace(list(orgsize_unknown), np.nan)
        .map(orgsize_map)
        .astype(float)
        / max_val
    )

edlevel_order = [
    "primary/elementary school",
    "secondary school",
    "some college/university study without earning a degree",
    "associate degree",
    "bachelor’s degree",
    "master’s degree",
    "professional degree",
]
edlevel_unknown = {"other"}

if "EdLevel" in df.columns:
    edlevel_map = {cat: i for i, cat in enumerate(edlevel_order)}
    max_val = len(edlevel_order) - 1

    df["EdLevel"] = (
        df["EdLevel"]
        .replace(list(edlevel_unknown), np.nan)
        .map(edlevel_map)
        .astype(float)
        / max_val
    )

# ---------- Kontrolle: Welche Werte konnten nicht gemappt werden? ----------
for col, unknown_set in [("OrgSize", orgsize_unknown), ("EdLevel", edlevel_unknown)]:
    if col in df.columns:
        unmapped = (
            df.loc[df[col].isna(), col]  # ist nach Mapping NaN
        )
# (Optional) numerische Übersicht
df[["OrgSize", "EdLevel"]].describe(include="all")


,OrgSize,EdLevel
count,17010.000000,18427.000000
mean,0.480810,0.663402
std,0.297300,0.201638
min,0.000000,0.000000
25%,0.285714,0.666667
50%,0.428571,0.666667
75%,0.714286,0.833333
max,1.000000,1.000000


In [8]:
object_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("Object-Spalten:")
object_columns


Object-Spalten:


['MainBranch',
 'Age',
 'Employment',
 'Country',
 'LearnCodeAI',
 'DevType',
 'ICorPM',
 'RemoteWork',
 'Industry',
 'AIThreat',
 'NewRole',
 'LanguageChoice',
 'DatabaseChoice',
 'PlatformChoice',
 'WebframeChoice',
 'DevEnvsChoice',
 'AIModelsChoice',
 'AISelect',
 'AIAgents']

In [9]:
if object_columns:
    nunique_per_column = df[object_columns].nunique(dropna=True)
else:
    nunique_per_column = pd.Series(dtype=int)

nunique_per_column


MainBranch          6
Age                 5
Employment          5
Country           163
LearnCodeAI         5
DevType            31
ICorPM              2
RemoteWork          5
Industry           15
AIThreat            3
NewRole             5
LanguageChoice      2
DatabaseChoice      2
PlatformChoice      2
WebframeChoice      2
DevEnvsChoice       2
AIModelsChoice      2
AISelect            5
AIAgents            6
dtype: int64

In [10]:
columns_to_encode = nunique_per_column[nunique_per_column <= max_unique].index.tolist()

#exclude_columns = {'xxx'}

#columns_to_encode = [
#    col for col in columns_to_encode
#    if col not in exclude_columns
#]

high_cardinality_columns = nunique_per_column[nunique_per_column > max_unique].index.tolist()

print("Spalten für One-Hot-Encoding (≤ 50 Werte):")
print(columns_to_encode)

print("\nSpalten mit hoher Kardinalität (> 50 Werte):")
high_cardinality_columns


# EducationLevel und OrgSize wurden oben ordinal encodiert -> nicht one-hot encodieren
exclude_cols = [c for c in ['EdLevel','OrgSize'] if c in df.columns]
columns_to_encode = [c for c in columns_to_encode if c not in exclude_cols]
high_cardinality_columns = [c for c in high_cardinality_columns if c not in exclude_cols]


Spalten für One-Hot-Encoding (≤ 50 Werte):
['MainBranch', 'Age', 'Employment', 'Country', 'LearnCodeAI', 'DevType', 'ICorPM', 'RemoteWork', 'Industry', 'AIThreat', 'NewRole', 'LanguageChoice', 'DatabaseChoice', 'PlatformChoice', 'WebframeChoice', 'DevEnvsChoice', 'AIModelsChoice', 'AISelect', 'AIAgents']

Spalten mit hoher Kardinalität (> 50 Werte):


In [11]:
df_encoded = pd.get_dummies(
    df,
    columns=columns_to_encode,
    dummy_na=na_as_category,
    drop_first=False
)

print("Neue DataFrame-Form nach One-Hot-Encoding:", df_encoded.shape)
# df_encoded
choice_cols = [c for c in df_encoded.columns if "Choice" in c]

print(f"Dropping Choice columns: {len(choice_cols)}")
# optional: zeig ein paar Beispiele
print("Examples:", choice_cols[:20])

df_encoded = df_encoded.drop(columns=choice_cols)

print("Form nach Drop Choice:", df_encoded.shape)

Neue DataFrame-Form nach One-Hot-Encoding: (18603, 405)
Dropping Choice columns: 18
Examples: ['LanguageChoice_no', 'LanguageChoice_yes', 'LanguageChoice_nan', 'DatabaseChoice_no', 'DatabaseChoice_yes', 'DatabaseChoice_nan', 'PlatformChoice_no', 'PlatformChoice_yes', 'PlatformChoice_nan', 'WebframeChoice_no', 'WebframeChoice_yes', 'WebframeChoice_nan', 'DevEnvsChoice_no', 'DevEnvsChoice_yes', 'DevEnvsChoice_nan', 'AIModelsChoice_no', 'AIModelsChoice_yes', 'AIModelsChoice_nan']
Form nach Drop Choice: (18603, 387)


In [12]:
if high_cardinality_columns:
    print("Nicht encodierte Spalten mit mehr als", max_unique, "verschiedenen Einträgen:")
    for col in high_cardinality_columns:
        print(f" - {col}: {int(nunique_per_column[col])} unique values")
else:
    print("Keine Spalten mit hoher Kardinalität gefunden.")


Keine Spalten mit hoher Kardinalität gefunden.


In [13]:
df_encoded.to_csv("One-Hot-Encoded-final.csv", index=False)

In [14]:
df_encoded.shape

(18603, 387)

In [15]:
# multiselect_cols:
#     "LanguageHaveWorkedWith",
#     "DatabaseHaveWorkedWith",
#     "PlatformHaveWorkedWith",
#     "WebframeHaveWorkedWith",
#     "DevEnvsHaveWorkedWith",
#     "OfficeStackAsyncHaveWorkedWith",
#     "AIModelsHaveWorkedWith",

lang_cols = [c for c in df_encoded.columns if c.startswith("PlatformChoice")]
print(f"Gefunden: {len(lang_cols)} Spalten")
print(*lang_cols, sep="\n")

Gefunden: 0 Spalten

